<a href="https://colab.research.google.com/github/prarthanadaibagya1/pyt/blob/main/lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lab 10:
Automated Multi-Format Sales Report Pipeline and Delivery


Objectives:

-to design and implement a production-ready automated report pipeline that extracts, transforms, and generates sales reports in multiple formats (Excel, HTML, PDF) and optionally delivers them via email.


In [2]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=311e0907c53daa9a1772c9210f800371fb300b24c0c41024e177902af1c4ea67
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


1. Extract → Transform → Generate → Deliver:
- Extract: Load raw sales data.
- Transform: Clean data, calculate revenue, aggregate summaries.
- Generate: Create reports in Excel, HTML, PDF.
- Deliver: Send report via email.

2. Ensuring Consistency Across Reports:
All reports (Excel, HTML, PDF) are generated from the same transformed dataset (summary DataFrame), so values like total revenue and units remain identical in every format.

3. _

4. Securing Email Credentials:
- Use environment variables instead of hardcoding passwords.
- Use App Passwords (Gmail) instead of main password





In [4]:
def load_data():
    data = {
        "Month": ["Jan", "Jan", "Feb", "Feb", "Mar", "Mar"],
        "Product": ["A", "B", "A", "B", "A", "C"],
        "Units Sold": [100, 150, 200, 120, 180, 130],
        "Unit Price": [10, 15, 10, 15, 12, 20],
        "Region": ["East", "West", "East", "West", "East", "North"]
    }
    return pd.DataFrame(data)

In [5]:
def transform_data(df):
    df["Revenue"] = df["Units Sold"] * df["Unit Price"]

    summary = df.groupby("Product").agg(
        Total_Units=("Units Sold", "sum"),
        Total_Revenue=("Revenue", "sum"),
        Avg_Price=("Unit Price", "mean")
    ).reset_index()

    total_revenue = summary["Total_Revenue"].sum()
    summary["Revenue_Share_%"] = (summary["Total_Revenue"] / total_revenue) * 100

    monthly = df.groupby("Month")["Revenue"].sum().reset_index()

    return df, summary, monthly


In [6]:
def generate_excel(summary, monthly, filename):
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        summary.to_excel(writer, sheet_name='Summary', index=False)
        monthly.to_excel(writer, sheet_name='Monthly Revenue', index=False)

In [18]:
def generate_html(summary, total_revenue, filename):
    template = Template("""
    <html>
    <head><title>Sales Report</title></head>
    <body>
        <h2>Sales Summary</h2>
        <table border="1">
            <tr>
                <th>Product</th><th>Total Units</th><th>Total Revenue</th><th>Avg Price</th><th>Revenue Share (%)</th>
            </tr>
            {% for row in data %}
            <tr>
                <td>{{row.Product}}</td>
                <td>{{row.Total_Units}}</td>
                <td>{{row.Total_Revenue}}</td>
                <td>{{row.Avg_Price}}</td>
                <td>{{"%.2f" % row['Revenue_Share_%']}}</td>
            </tr>
            {% endfor %}
        </table>
        <h3>Total Revenue: {{total}}</h3>
    </body>
    </html>
    """)
    html = template.render(data=summary.to_dict(orient='records'), total=total_revenue)

    with open(filename, 'w') as f:
        f.write(html)

In [28]:
class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 10, 'Sales Report', 0, 1, 'C')


def generate_pdf(summary, filename):
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Arial", size=10)

    # Table header
    headers = list(summary.columns)
    for col in headers:
        pdf.cell(38, 8, col, border=1)
    pdf.ln()
 # Rows
    for i, row in summary.iterrows():
        for item in row:
            pdf.cell(38, 8, str(round(item,2)) if isinstance(item, float) else str(item), border=1)
        pdf.ln()
    # ---- Generate and add bar chart ----
    plt.bar(monthly['Month'], monthly['Revenue'], color='skyblue')
    plt.title('Monthly Revenue')
    plt.xlabel('Month')
    plt.ylabel('Revenue')
    plt.tight_layout()
    chart_file = "chart.png"
    plt.savefig(chart_file)
    plt.close()

    # Insert chart into PDF
    pdf.add_page()  # add a new page for chart
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, 'Monthly Revenue Chart', 0, 1, 'C')
    pdf.image(chart_file, x=10, y=30, w=180)

    pdf.output(filename)

In [9]:
def send_email(sender, password, receiver, file_path):
    msg = EmailMessage()
    msg['Subject'] = 'Sales Report'
    msg['From'] = sender
    msg['To'] = receiver
    msg.set_content('Please find attached the sales report.')

    with open(file_path, 'rb') as f:
        msg.add_attachment(f.read(), maintype='application', subtype='octet-stream', filename=os.path.basename(file_path))

    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        smtp.login(sender, password)
        smtp.send_message(msg)

In [31]:
import pandas as pd
import os
from datetime import datetime
from jinja2 import Template
from fpdf import FPDF
import smtplib
from email.message import EmailMessage
import matplotlib.pyplot as plt

In [32]:
class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 10, 'Sales Report', 0, 1, 'C')


def generate_pdf(summary, monthly, filename):
    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Arial", size=10)

    # Table header
    headers = list(summary.columns)
    for col in headers:
        pdf.cell(38, 8, col, border=1)
    pdf.ln()
 # Rows
    for i, row in summary.iterrows():
        for item in row:
            pdf.cell(38, 8, str(round(item,2)) if isinstance(item, float) else str(item), border=1)
        pdf.ln()
    # ---- Generate and add bar chart ----
    plt.bar(monthly['Month'], monthly['Revenue'], color='skyblue')
    plt.title('Monthly Revenue')
    plt.xlabel('Month')
    plt.ylabel('Revenue')
    plt.tight_layout()
    chart_file = "chart.png"
    plt.savefig(chart_file)
    plt.close()

    # Insert chart into PDF
    pdf.add_page()  # add a new page for chart
    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, 'Monthly Revenue Chart', 0, 1, 'C')
    pdf.image(chart_file, x=10, y=30, w=180)

    pdf.output(filename)

In [33]:
def main():
    df = load_data()
    df, summary, monthly = transform_data(df)

    today = datetime.now().strftime("%Y-%m-%d")
    os.makedirs("reports", exist_ok=True)

    excel_file = f"reports/report_{today}.xlsx"
    html_file = f"reports/report_{today}.html"
    pdf_file = f"reports/report_{today}.pdf"

    generate_excel(summary, monthly, excel_file)
    generate_html(summary, summary['Total_Revenue'].sum(), html_file)
    generate_pdf(summary, monthly, pdf_file)

    # Optional Email
    # send_email("your_email@gmail.com", "your_app_password", "receiver@gmail.com", pdf_file)

    print("Reports generated successfully!")


if __name__ == "__main__":
    main()

Reports generated successfully!


In [29]:
import os
os.makedirs("reports", exist_ok=True)

In [23]:
!ls reports/

report_2026-03-18.html	report_2026-03-18.pdf  report_2026-03-18.xlsx
